# TAILOR quickstart: age-optimal preemption under heavy-tailed delays

Companion notebook for *Taming the Heavy Tail: Age-Optimal Preemption* (A. Li, Y. İnce, E. Uysal, IEEE ISIT 2026, [DOI](https://doi.org/10.1109/ISIT62367.2026.11653616), [arXiv](https://arxiv.org/abs/2601.16624)).

We compute the optimal joint **sampling + preemption** policy for a status-update link with random service delay, and compare it with non-preemptive baselines.

In [ ]:
!pip -q install "tailor-aoi[plot]"

## 1. Pick a service-time distribution
Any distribution with $\mathbb{E}[Y^2]<\infty$ works: Pareto II (Lomax), log-normal, exponential, or any `scipy.stats` law via `ServiceDist.from_scipy`.

In [ ]:
from tailor_aoi import ServiceDist, solve, compare, simulate, sample_path

dist = ServiceDist.lognormal_from_mean(mean=2.0, sigma2=4.0)
dist

## 2. Solve for the optimal policy
`solve` runs TAILOR's average-cost policy iteration on the integral ACOEs (Theorem 1). It returns the optimal average cost $\rho$, the sampling rule $z^\star(\Delta)$ and the preemption rule $\theta^\star(y)$.

In [ ]:
policy = solve(dist, kappa_s=1.0, kappa_p=1.0)
print(policy.summary())
print(compare(dist, 1.0, 1.0, policy=policy))

## 3. Inspect the policy

In [ ]:
from tailor_aoi.plotting import plot_policy, plot_sample_path
plot_policy(policy, y_max=10);

## 4. Verify by simulation
The simulator follows the SMDP dynamics exactly and reports time averages and the 99% AoI quantile.

In [ ]:
print(simulate(policy, dist, 1.0, 1.0, n_attempts=300_000, rng=0))
ts, ages, pre = sample_path(policy, dist, horizon=40, rng=1)
plot_sample_path(ts, ages, pre, label='TAILOR');

## 5. Variability becomes an advantage
As $\sigma^2$ grows, TAILOR's cost *decreases* while the non-preemptive optimum degrades (Fig. 2 of the paper; baselines here are exact).

In [ ]:
from tailor_aoi import update_or_wait
for s2 in [1, 2, 3, 4, 5, 6]:
    d = ServiceDist.lognormal_from_mean(2.0, s2)
    r, u = solve(d, 1, 1).rho, update_or_wait(d, 1, 1)[1]
    print(f'sigma^2={s2}:  TAILOR {r:6.3f}   Update-or-Wait {u:8.3f}   gain {u/r:5.1f}x')

## Cite
If TAILOR helps your research, please cite the paper:

In [ ]:
from tailor_aoi import cite
print(cite())